# Classifying Handwritten Digits with CNNs
**Daily Challenge — MNIST Dataset**

In this notebook you will:
1. Load and explore the MNIST dataset
2. Build and train a **Fully Connected Neural Network**
3. Build and train a **Convolutional Neural Network (CNN)**
4. Compare the performance of both architectures


## 1. Imports & Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow version:", tf.__version__)


## 2. Load the MNIST Dataset

In [ ]:
(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = keras.datasets.mnist.load_data()

print(f"X_train shape : {X_train_raw.shape}")
print(f"y_train shape : {y_train_raw.shape}")
print(f"X_test  shape : {X_test_raw.shape}")
print(f"y_test  shape : {y_test_raw.shape}")
print(f"Pixel range   : [{X_train_raw.min()}, {X_train_raw.max()}]")
print(f"Classes       : {np.unique(y_train_raw)}")


### Sample Images

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for i in range(20):
    ax = axes[i // 10][i % 10]
    ax.imshow(X_train_raw[i], cmap="gray")
    ax.set_title(f"Label: {y_train_raw[i]}", fontsize=8)
    ax.axis("off")
plt.suptitle("Sample MNIST Images", fontsize=12)
plt.tight_layout()
plt.show()


## 3. Fully Connected Neural Network

### 3a. Preprocess for FC Network
Flatten 28×28 images → 784-dimensional vectors, normalise to [0, 1], and one-hot encode labels.


In [ ]:
# Flatten and normalise
X_train_fc = X_train_raw.reshape(-1, 784).astype("float32") / 255.0
X_test_fc  = X_test_raw.reshape(-1, 784).astype("float32") / 255.0

# One-hot encode
y_train_ohe = keras.utils.to_categorical(y_train_raw, num_classes=10)
y_test_ohe  = keras.utils.to_categorical(y_test_raw,  num_classes=10)

print(f"X_train_fc   : {X_train_fc.shape}  (flattened & normalised)")
print(f"y_train_ohe  : {y_train_ohe.shape}  (one-hot encoded)")
print(f"Example label: {y_train_raw[0]} → {y_train_ohe[0]}")


### 3b. Build the FC Model

In [ ]:
fc_model = keras.Sequential([
    keras.layers.Input(shape=(784,), name="input"),
    keras.layers.Dense(512, activation="relu",    name="dense_1"),
    keras.layers.Dense(256, activation="relu",    name="dense_2"),
    keras.layers.Dense(128, activation="relu",    name="dense_3"),
    keras.layers.Dense(10,  activation="softmax", name="output"),
], name="FullyConnectedNN")

fc_model.summary()


### 3c. Compile & Train

In [ ]:
fc_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

fc_history = fc_model.fit(
    X_train_fc, y_train_ohe,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    verbose=1,
)


In [ ]:
fc_loss, fc_accuracy = fc_model.evaluate(X_test_fc, y_test_ohe, verbose=0)
print(f"FC Model — Test Loss: {fc_loss:.4f} | Test Accuracy: {fc_accuracy * 100:.2f}%")


## 4. Convolutional Neural Network (CNN)

### 4a. Preprocess for CNN
Reshape to `(N, 28, 28, 1)` — Conv2D layers need a channel dimension.


In [ ]:
X_train_cnn = X_train_raw.reshape(-1, 28, 28, 1).astype("float32") / 255.0
X_test_cnn  = X_test_raw.reshape(-1, 28, 28, 1).astype("float32") / 255.0

print(f"X_train_cnn : {X_train_cnn.shape}  (H × W × channels)")
print(f"X_test_cnn  : {X_test_cnn.shape}")


### 4b. Build the CNN

In [ ]:
cnn_model = keras.Sequential([
    keras.layers.Input(shape=(28, 28, 1), name="input"),

    # Block 1 — 32 filters, 3×3 kernel
    keras.layers.Conv2D(32, kernel_size=(3, 3), activation="relu", padding="same", name="conv_1"),
    keras.layers.MaxPooling2D(pool_size=(2, 2), name="pool_1"),

    # Block 2 — 64 filters, 3×3 kernel
    keras.layers.Conv2D(64, kernel_size=(3, 3), activation="relu", padding="same", name="conv_2"),
    keras.layers.MaxPooling2D(pool_size=(2, 2), name="pool_2"),

    # Classifier head
    keras.layers.Flatten(name="flatten"),
    keras.layers.Dense(128, activation="relu",    name="dense_1"),
    keras.layers.Dense(10,  activation="softmax", name="output"),
], name="CNN")

cnn_model.summary()


### 4c. Compile & Train

In [ ]:
cnn_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

cnn_history = cnn_model.fit(
    X_train_cnn, y_train_ohe,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    verbose=1,
)


In [ ]:
cnn_loss, cnn_accuracy = cnn_model.evaluate(X_test_cnn, y_test_ohe, verbose=0)
print(f"CNN Model — Test Loss: {cnn_loss:.4f} | Test Accuracy: {cnn_accuracy * 100:.2f}%")


## 5. Performance Comparison

In [ ]:
# ── Accuracy & Loss curves ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Accuracy
axes[0].plot(fc_history.history["val_accuracy"],  label="FC NN (val)",  linestyle="--", marker="o")
axes[0].plot(cnn_history.history["val_accuracy"], label="CNN (val)",    linestyle="--", marker="s")
axes[0].plot(fc_history.history["accuracy"],      label="FC NN (train)",  alpha=0.5)
axes[0].plot(cnn_history.history["accuracy"],     label="CNN (train)",    alpha=0.5)
axes[0].set_title("Accuracy per Epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(fc_history.history["val_loss"],  label="FC NN (val)",  linestyle="--", marker="o")
axes[1].plot(cnn_history.history["val_loss"], label="CNN (val)",    linestyle="--", marker="s")
axes[1].plot(fc_history.history["loss"],      label="FC NN (train)",  alpha=0.5)
axes[1].plot(cnn_history.history["loss"],     label="CNN (train)",    alpha=0.5)
axes[1].set_title("Loss per Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
fc_params  = fc_model.count_params()
cnn_params = cnn_model.count_params()
improvement = (cnn_accuracy - fc_accuracy) * 100

print(f"{'Model':<28} {'Test Accuracy':>14} {'Test Loss':>10} {'Parameters':>12}")
print("-" * 68)
print(f"{'Fully Connected NN':<28} {fc_accuracy * 100:>13.2f}% {fc_loss:>10.4f} {fc_params:>12,}")
print(f"{'CNN':<28} {cnn_accuracy * 100:>13.2f}% {cnn_loss:>10.4f} {cnn_params:>12,}")
print("-" * 68)
print(f"\n📈 CNN improvement: {improvement:+.2f} percentage points")
print("\n🔍 Key Takeaways:")
print("  • CNNs exploit spatial structure — nearby pixels are related.")
print("  • Conv2D + MaxPool2D extract hierarchical local features automatically.")
print("  • CNNs achieve higher accuracy with fewer parameters on image tasks.")
print("  • The Flatten layer bridges spatial feature maps to Dense classifiers.")
